In [2]:
import torch
import torch.nn as nn
import torchaudio
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os


class ESC50Dataset(Dataset):
    def __init__(self, csv_path, audio_dir, transform=None):
        self.data = pd.read_csv(csv_path)
        self.audio_dir = audio_dir
        self.transform = transform
        self.mel_spec = torchaudio.transforms.MelSpectrogram(sample_rate=44100, n_mels=128)

        # Build a mapping from class name to integer
        self.label2idx = {label: idx for idx, label in enumerate(sorted(self.data["classID"].unique()))}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row["slice_file_name"])
        waveform, sr = torchaudio.load(audio_path)

        mel = self.mel_spec(waveform)
        mel = torchaudio.functional.amplitude_to_DB(mel, multiplier=10, amin=1e-10, db_multiplier=0)

        if self.transform:
            mel = self.transform(mel)

        label_name = row["classID"]
        label_idx = self.label2idx[label_name]
        label_tensor = torch.tensor(label_idx, dtype=torch.long)

        return mel, label_tensor




In [3]:
import torchvision.transforms.functional as TF
transform = transforms.Compose([
    transforms.Lambda(lambda x: TF.resize(x, [224, 224])),  # Resize 2D spectrogram
])

dataset = ESC50Dataset(
    "/media/arafat/New Volume/UrbanSound8K/UrbanSound8K/metadata/UrbanSound8K.csv",
    "/media/arafat/New Volume/UrbanSound8K/UrbanSound8K/audio/all",
    transform,
)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

/home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [4]:
# Model
model = models.resnet18(pretrained=True)
model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)  # For 1-channel input
model.fc = nn.Linear(model.fc.in_features, 10)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training loop (simplified)
from torchmetrics import Accuracy  # Optional, or use manual method
import torch

# Optional: move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item() * inputs.size(0)

        # Accuracy calculation
        _, predicted = torch.max(outputs, 1)   # Get predicted class indices
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total * 100

    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f} | Accuracy = {accuracy:.2f}%")


/home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


RuntimeError: Failed to open the input "/media/arafat/New Volume/UrbanSound8K/UrbanSound8K/audio/all/99179-9-0-17.wav" (No such file or directory).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x57 (0x7f1db41f8897 in /home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7f1db41a8b25 in /home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42334 (0x7f1cbf1a6334 in /home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torio/lib/libtorio_ffmpeg4.so)
frame #3: torio::io::StreamingMediaDecoder::StreamingMediaDecoder(std::string const&, std::optional<std::string> const&, std::optional<std::map<std::string, std: